# SaansCare – Model 4: Route Health Risk Classifier Pipeline (core contribution)
**Goal:** classify a whole journey as **Safe / Caution / Avoid** from route-level pollution and travel features, instead of one raw AQI number.
**Algorithms:** **MLP (Multi-Layer Perceptron neural network, main model)**, with Random Forest, XGBoost and Logistic Regression as baselines.
**Metrics:** accuracy, precision, recall, per-class F1, macro-F1, confusion matrix, and **under-warning rate** (calling an Avoid route Safe).

**How to run:** run the cells top to bottom. Step 2 asks you for `route_health_risk.csv` / `.xlsx`.

## 1. Setup
*What we are doing:* importing libraries and creating the output folder.

In [ ]:
import sys, subprocess, os, json, warnings, zipfile, time
from pathlib import Path
warnings.filterwarnings("ignore")

def _ensure(pkg, import_name=None):
    """Install a package only if it is missing (Google Colab already has most of them)."""
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("xgboost"); _ensure("openpyxl"); _ensure("joblib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, mean_squared_error, mean_absolute_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42                      # fixed seed -> reproducible results
np.random.seed(RANDOM_STATE)
OUT_DIR = Path("saanscare_outputs/model4_route_risk")   # everything this notebook produces is saved here
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})
print("Setup complete. Outputs will be saved in:", OUT_DIR.resolve())

In [ ]:
def plot_confusion(y_true, y_pred, labels, title, path=None):
    """Confusion matrix as counts and as row-percentages (recall per class)."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(5.8, 5))
    ax.imshow(cmn, cmap="Greens", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i, j]:,}\n{cmn[i, j]*100:.0f}%", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.55 else "black", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title); ax.grid(False)
    plt.tight_layout()
    if path: plt.savefig(path, dpi=130)
    plt.show()

def warning_rates(y_true, y_pred):
    """For ordered risk classes: under-warning (predicted LOWER risk than reality) is the dangerous error."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {"exact": float((y_pred == y_true).mean()),
            "under_warning": float((y_pred < y_true).mean()),
            "over_warning": float((y_pred > y_true).mean())}

def clf_scores(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)}

def build_pre(num_cols, cat_cols, scale=True):
    """Preprocessing: impute missing values, scale numbers, one-hot encode text categories."""
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale: num_steps.append(("sc", StandardScaler()))
    cat_steps = [("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]
    return ColumnTransformer([("num", Pipeline(num_steps), num_cols), ("cat", Pipeline(cat_steps), cat_cols)])

def to_jsonable(o):
    if isinstance(o, dict): return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(to_jsonable(obj), f, indent=2)

def package_outputs(out_dir, zip_name):
    """Zip the output folder; in Google Colab this also starts the download."""
    zpath = Path(zip_name)
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in Path(out_dir).rglob("*"):
            if p.is_file(): z.write(p, p.relative_to(Path(out_dir).parent))
    print("Packaged:", zpath.resolve(), f"({zpath.stat().st_size/1e6:.1f} MB)")
    try:
        from google.colab import files
        files.download(str(zpath))
    except ImportError:
        pass

## 2. Load the dataset (the notebook asks you for it)
*What we are doing:* asking for the dataset file and checking that the expected columns exist.

In [ ]:
def load_dataset(label, required_cols):
    """Ask the user for the dataset, read it, and check that the expected columns are present."""
    path = os.environ.get("SAANSCARE_DATASET")            # optional shortcut: skips the prompt
    if not path:
        try:
            from google.colab import files                 # running in Google Colab -> upload dialog
            print(f"Upload your {label} file (.csv or .xlsx) ...")
            uploaded = files.upload()
            if not uploaded:
                raise FileNotFoundError("No file was uploaded.")
            path = next(iter(uploaded))
        except ImportError:                                # running in VS Code / Jupyter -> ask for the path
            path = input(f"Enter the full path of your {label} file (.csv or .xlsx): ").strip().strip('"').strip("'")
    df = pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"The file is missing these required columns: {missing}\nColumns found: {list(df.columns)}")
    print(f"Loaded {Path(path).name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df

REQUIRED = ["hour", "month", "origin_zone", "dest_zone", "travel_mode", "distance_km", "duration_min", "avg_aqi", "max_aqi", "aqi_std",
            "pct_segments_aqi_over_150", "traffic_density", "pct_arterial_roads", "n_signals", "pct_near_industrial", "green_cover_pct",
            "wind_speed", "temperature", "humidity", "user_group", "route_label"]
df = load_dataset("route health risk dataset", REQUIRED)
df.head()

## 3. Data understanding (EDA)
*What we are doing:* checking class balance and how the label changes with the month, travel mode and user group.

In [ ]:
LABELS = ["Safe", "Caution", "Avoid"]
print("Missing values:\n", df.isna().sum()[lambda s: s > 0].to_string() or "none")
print("\nLabel distribution (%):"); print((df["route_label"].value_counts(normalize=True).reindex(LABELS) * 100).round(1).to_string())
display(df.describe().T.round(2))
cols = ["#86EFAC", "#FDE047", "#DC2626"]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
pd.crosstab(df["month"], df["route_label"], normalize="index")[LABELS].plot(kind="bar", stacked=True, ax=ax[0], color=cols, title="Label by month")
pd.crosstab(df["travel_mode"], df["route_label"], normalize="index")[LABELS].plot(kind="bar", stacked=True, ax=ax[1], color=cols, title="Label by travel mode")
pd.crosstab(df["user_group"], df["route_label"], normalize="index")[LABELS].plot(kind="bar", stacked=True, ax=ax[2], color=cols, title="Label by user group")
plt.tight_layout(); plt.show()

## 4. Data cleaning
*What we are doing:* removing duplicates and impossible values, clipping extreme outliers, and dropping columns that must **not** be inputs:
the exposure score and noise flag (they are the label's own ingredients), the id and date, and the simulated "cleanest route" estimate.

In [ ]:
n0 = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "route_id"]).reset_index(drop=True)
print("Duplicates removed:", n0 - len(df))
valid = ((df["distance_km"] > 0) & (df["duration_min"] > 0) & df["avg_aqi"].between(0, 500) & df["hour"].between(0, 23)
         & df["month"].between(1, 12) & df["route_label"].isin(LABELS))
print("Invalid rows dropped:", int((~valid).sum()))
df = df[valid].reset_index(drop=True)

EXCLUDE = [c for c in ["route_id", "date", "exposure_score", "label_noise_applied", "fastest_vs_cleanest_exposure_reduction_pct"] if c in df.columns]
print("Excluded from inputs (leakage / not available at prediction time):", EXCLUDE)
for col in ["distance_km", "duration_min"]:
    df[col] = df[col].clip(upper=df[col].quantile(0.999))
for col in ["origin_zone", "dest_zone", "travel_mode", "user_group"]:
    df[col] = df[col].astype(str).str.strip()
print("Clean shape:", df.shape)

## 5. Feature engineering
*What we are doing:* adding useful derived inputs: average speed, rush-hour flag, a same-zone flag, and hour/month as smooth sin/cos waves.

In [ ]:
def add_features(d):
    d = d.copy()
    d["avg_speed_kmh"] = d["distance_km"] / (d["duration_min"] / 60)
    d["is_rush_hour"] = d["hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)
    d["same_zone"] = (d["origin_zone"] == d["dest_zone"]).astype(int)
    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24); d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
    d["month_sin"] = np.sin(2 * np.pi * d["month"] / 12); d["month_cos"] = np.cos(2 * np.pi * d["month"] / 12)
    return d

df = add_features(df)
CAT = ["origin_zone", "dest_zone", "travel_mode", "user_group"]
NUM = ["distance_km", "duration_min", "avg_aqi", "max_aqi", "aqi_std", "pct_segments_aqi_over_150", "traffic_density", "pct_arterial_roads",
       "n_signals", "pct_near_industrial", "green_cover_pct", "wind_speed", "temperature", "humidity", "avg_speed_kmh", "is_rush_hour", "same_zone",
       "hour_sin", "hour_cos", "month_sin", "month_cos"]
FEATURES = NUM + CAT
LABEL_TO_ID = {l: i for i, l in enumerate(LABELS)}
X = df[FEATURES]; y = df["route_label"].map(LABEL_TO_ID)
print(len(FEATURES), "input features")

## 6. Train / validation / test split (stratified)
*What we are doing:* 70% / 15% / 15% stratified on the label. The validation set is used to choose settings; the test set is used once at the end.

In [ ]:
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=RANDOM_STATE)
print(f"train {len(Xtr):,} | validation {len(Xva):,} | test {len(Xte):,}")
w_tr = compute_sample_weight("balanced", ytr)

def oversample(Xd, yd, seed=RANDOM_STATE):
    """Balance the TRAINING set by repeating minority-class rows (MLP has no class_weight option). Never applied to val/test."""
    rng_ = np.random.default_rng(seed); target = yd.value_counts().max(); idx = []
    for cls_, n_ in yd.value_counts().items():
        pos = np.where(yd.values == cls_)[0]
        idx.extend(rng_.choice(pos, target, replace=n_ < target) if n_ < target else pos)
    idx = rng_.permutation(np.array(idx))
    return Xd.iloc[idx], yd.iloc[idx]
Xtr_bal, ytr_bal = oversample(Xtr, ytr)
print("Balanced training set for the MLP:", ytr_bal.value_counts().sort_index().to_dict())

## 7. Preprocessing pipeline
*What we are doing:* fill missing values, **scale** numbers (essential for the neural network), one-hot encode zones/mode/user group. Saved inside the model.

In [ ]:
pre_scaled = build_pre(NUM, CAT, scale=True)
pre_trees = build_pre(NUM, CAT, scale=False)
print("Preprocessing ready")

## 8. Model training and comparison
*What we are doing:* training the **MLP** (main model) and three baselines. The MLP is a neural network: input layer -> hidden layers (ReLU) -> 3 outputs (Safe/Caution/Avoid), trained with early stopping.

In [ ]:
def make_mlp(hidden=(128, 64), alpha=1e-4, lr=1e-3, batch=128):
    return Pipeline([("pre", pre_scaled), ("model", MLPClassifier(hidden_layer_sizes=hidden, activation="relu", alpha=alpha, learning_rate_init=lr,
                                                                    batch_size=batch, early_stopping=True, validation_fraction=0.1, n_iter_no_change=12,
                                                                    max_iter=300, random_state=RANDOM_STATE))])
models = {
    "MLP (main)": make_mlp(),
    "Random Forest": Pipeline([("pre", pre_trees), ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE))]),
    "XGBoost": Pipeline([("pre", pre_trees), ("model", XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
                                                                      objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE))]),
    "Logistic Regression": Pipeline([("pre", pre_scaled), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
}
rows = []
for name, pipe in models.items():
    t0 = time.time()
    if name == "MLP (main)": pipe.fit(Xtr_bal, ytr_bal)
    elif name == "XGBoost": pipe.fit(Xtr, ytr, model__sample_weight=w_tr)
    else: pipe.fit(Xtr, ytr)
    pv = pipe.predict(Xva); s = clf_scores(yva, pv); s.update(warning_rates(yva, pv)); s["seconds"] = round(time.time() - t0, 1)
    rows.append(pd.Series(s, name=name))
comparison = pd.DataFrame(rows).round(3).sort_values("f1_macro", ascending=False)
comparison

## 9. Hyperparameter tuning (MLP, XGBoost, Random Forest)
*What we are doing:* trying random settings and scoring each on the validation set (macro-F1). For the MLP we search network size, regularisation and learning rate;
for the tree models, depth, learning rate, number of trees.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
mlp_space = {"hidden": [(64,), (128, 64), (128, 64, 32), (256, 128), (256, 128, 64)], "alpha": [1e-5, 1e-4, 1e-3, 1e-2],
             "lr": [3e-4, 1e-3, 3e-3], "batch": [64, 128, 256]}
trials = []
t0 = time.time()
for k in range(8):
    p = {n_: v[rng.integers(len(v))] for n_, v in mlp_space.items()}
    m = make_mlp(**p).fit(Xtr_bal, ytr_bal)
    trials.append({**{n_: str(v) for n_, v in p.items()}, "val_f1_macro": f1_score(yva, m.predict(Xva), average="macro"), "iters": m.named_steps["model"].n_iter_})
mlp_trials = pd.DataFrame(trials).sort_values("val_f1_macro", ascending=False).round(4)
print(f"MLP search took {time.time()-t0:.0f}s"); display(mlp_trials)
bp = mlp_trials.iloc[0]
best_mlp_params = {"hidden": eval(bp["hidden"]), "alpha": float(bp["alpha"]), "lr": float(bp["lr"]), "batch": int(bp["batch"])}
best_mlp = make_mlp(**best_mlp_params).fit(Xtr_bal, ytr_bal)

xgb_s = RandomizedSearchCV(models["XGBoost"], {"model__n_estimators": [200, 300, 500], "model__max_depth": [4, 6, 8], "model__learning_rate": [0.05, 0.1, 0.2],
                           "model__subsample": [0.8, 0.9, 1.0], "model__colsample_bytree": [0.7, 0.9, 1.0]}, n_iter=8, cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                           scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=1).fit(Xtr, ytr, model__sample_weight=w_tr)
rf_s = RandomizedSearchCV(models["Random Forest"], {"model__n_estimators": [200, 400], "model__max_depth": [None, 15, 25], "model__min_samples_leaf": [1, 2, 4], "model__max_features": ["sqrt", 0.5]},
                          n_iter=5, cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE), scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=1).fit(Xtr, ytr)
tuned = {"MLP (tuned)": best_mlp, "XGBoost (tuned)": xgb_s.best_estimator_, "Random Forest (tuned)": rf_s.best_estimator_}
tuned_rows = {n_: {**clf_scores(yva, m_.predict(Xva)), **warning_rates(yva, m_.predict(Xva))} for n_, m_ in tuned.items()}
tuned_df = pd.DataFrame(tuned_rows).T.round(3).sort_values("f1_macro", ascending=False); display(tuned_df)

## 10. MLP training curve
*What we are doing:* checking that the neural network learned properly: the loss should fall and the internal validation score should flatten, without a big gap (overfitting).

In [ ]:
mm = best_mlp.named_steps["model"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(mm.loss_curve_); ax[0].set_title("Training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(mm.validation_scores_); ax[1].set_title("Internal validation accuracy"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(OUT_DIR / "mlp_training_curve.png", dpi=130); plt.show()
print("Stopped after", mm.n_iter_, "epochs; best settings:", best_mlp_params)

## 11. Final evaluation on the test set
*What we are doing:* the honest score. The **MLP** (main model) and the tuned baselines are evaluated once on unseen routes: accuracy, precision, recall, F1 per class, confusion matrix, under-warning rate.

In [ ]:
test_rows = {}
for n_, m_ in tuned.items():
    pt = m_.predict(Xte); test_rows[n_] = {**clf_scores(yte, pt), **warning_rates(yte, pt)}
test_df = pd.DataFrame(test_rows).T.round(3).sort_values("f1_macro", ascending=False); display(test_df)

pred = best_mlp.predict(Xte)
print("\nMLP (main model) – detailed report")
print(classification_report(yte, pred, target_names=LABELS, digits=3))
plot_confusion(yte, pred, LABELS, "Route risk – MLP (test)", OUT_DIR / "confusion_matrix_test_mlp.png")
print("Worst error (Avoid predicted as Safe):", int(((yte.values == 2) & (pred == 0)).sum()), "routes")
bd = pd.DataFrame({"mode": Xte["travel_mode"].values, "group": Xte["user_group"].values, "ok": pred == yte.values})
print("\nAccuracy by travel mode:"); display(bd.groupby("mode")["ok"].mean().round(3).to_frame("accuracy"))
print("Accuracy by user group:"); display(bd.groupby("group")["ok"].mean().round(3).to_frame("accuracy"))

## 12. Feature importance
*What we are doing:* permutation importance for the MLP: how much macro-F1 drops when one input is shuffled (a 3,000-route sample keeps it quick).

In [ ]:
sub = Xte.sample(min(3000, len(Xte)), random_state=RANDOM_STATE); ysub = yte.loc[sub.index]
pi = permutation_importance(best_mlp, sub, ysub, scoring="f1_macro", n_repeats=3, random_state=RANDOM_STATE, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=sub.columns).sort_values().tail(15)
imp.plot(kind="barh", color="#16A34A", title="MLP permutation importance (top 15)"); plt.tight_layout()
plt.savefig(OUT_DIR / "feature_importance_mlp.png", dpi=130); plt.show()

## 13. Save the model
*What we are doing:* saving the MLP pipeline as the production model, plus the best tree baseline for comparison, with metadata and metrics.

In [ ]:
best_tree_name = max(["XGBoost (tuned)", "Random Forest (tuned)"], key=lambda n_: test_rows[n_]["f1_macro"])
meta = {"model": "Route health risk classifier (MLP)", "labels": LABELS, "features": FEATURES, "num_features": NUM, "cat_features": CAT,
        "mlp_params": best_mlp_params, "metrics_test_mlp": test_rows["MLP (tuned)"], "baselines_test": {k: v for k, v in test_rows.items() if k != "MLP (tuned)"},
        "excluded_columns": EXCLUDE, "note": "Labels come from a documented exposure rule; trained on simulated routes unless retrained on real OSMnx routes.",
        "version": "1.0", "random_state": RANDOM_STATE}
joblib.dump({"pipeline": best_mlp, "meta": meta}, OUT_DIR / "route_risk_mlp.joblib")
joblib.dump({"pipeline": tuned[best_tree_name], "meta": {**meta, "model": f"Route risk baseline ({best_tree_name})"}}, OUT_DIR / "route_risk_baseline.joblib")
save_json(meta, OUT_DIR / "route_risk_metadata.json")
comparison.to_csv(OUT_DIR / "model_comparison_validation.csv"); mlp_trials.to_csv(OUT_DIR / "mlp_tuning_trials.csv", index=False); test_df.to_csv(OUT_DIR / "test_comparison.csv")
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## 14. Inference demo (how the web app will use it)
*What we are doing:* loading the saved model and classifying one route. In the app, the route optimizer (A*) computes these route features from the road graph and the AQI map, then calls this model.

In [ ]:
def classify_route(route: dict):
    b = joblib.load(OUT_DIR / "route_risk_mlp.joblib"); m = b["meta"]
    row = add_features(pd.DataFrame([route]))[m["features"]]
    p = b["pipeline"].predict_proba(row)[0]
    return {"route_label": m["labels"][int(p.argmax())], "probabilities": {l: round(float(x), 3) for l, x in zip(m["labels"], p)}}

example = {"hour": 8, "month": 11, "origin_zone": "Johar Town", "dest_zone": "Gulberg", "travel_mode": "bike", "distance_km": 9.5, "duration_min": 32,
           "avg_aqi": 260, "max_aqi": 340, "aqi_std": 40, "pct_segments_aqi_over_150": 95, "traffic_density": 0.8, "pct_arterial_roads": 55, "n_signals": 9,
           "pct_near_industrial": 10, "green_cover_pct": 8, "wind_speed": 1.2, "temperature": 14, "humidity": 70, "user_group": "sensitive"}
print(classify_route(example))
print(classify_route({**example, "month": 7, "avg_aqi": 70, "max_aqi": 110, "pct_segments_aqi_over_150": 0, "travel_mode": "car", "user_group": "general", "temperature": 34}))

## 15. Package and download
*What we are doing:* zipping the models, metadata, metrics and plots.

In [ ]:
package_outputs(OUT_DIR, "model4_route_risk_outputs.zip")

### Notes for the viva
* The label comes from a documented exposure rule (plus ~2% label noise), so the MLP learns that rule from route features; the scores prove the pipeline, not real-world accuracy.
* Retrain on real routes (OSMnx graph + AQI from Model 1 / CAMS) with the same columns. The `fastest_vs_cleanest...` column is excluded here: replace it with the real value from the A* optimizer and decide whether it is available at prediction time.
* Under-warning (Avoid -> Safe) is reported separately because it is the dangerous mistake.